# COMPSCI 4NL3 – Assignment 3: Word Embeddings
**Winter 2026 | Due: March 11th**

---
### ⚠️ AI / External Code Disclaimer
> **Include any disclaimers about the use of AI tools and cite the parts of the code that AI was used for.  
> Follow the syllabus instructions: if you use generative AI and do not report it, you may receive a 0 for the assignment.**

**Your Name / Student ID:** `TODO`  
**AI Tools Used:** `TODO (e.g. GitHub Copilot for boilerplate, ChatGPT for debugging)`


## 0. Setup

Install required packages and import libraries. Run this cell first every time you open the notebook.


In [ ]:


# Install packages (uncomment on first run or in Colab)
# !pip install datasets apache_beam gensim wefe scikit-learn pandas matplotlib seaborn

import os, random, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Setup complete. Gensim version:', gensim.__version__)


Setup complete. Gensim version: 4.4.0


---
## 1. Dataset Loading

We load **Simple English Wikipedia** from the HuggingFace Datasets hub.  
This is ~260k articles and is much more manageable than full English Wikipedia (~7M articles).

> **Tip (Colab users):** After training your models, download the `.model` files to avoid
> retraining every session.


In [2]:
from datasets import load_dataset

# Load Simple English Wikipedia
# This may take a few minutes the first time.
dataset = load_dataset('wikimedia/wikipedia', '20231101.simple')
# Inspect the first article
print('Keys:', dataset['train'].features)
print('\nFirst article title:', dataset['train'][0]['title'])
print('Text snippet:', dataset['train'][0]['text'][:300])
print('\nTotal articles:', len(dataset['train']))


Generating train split: 100%|██████████| 241787/241787 [00:00<00:00, 303945.15 examples/s]

Keys: {'id': Value('string'), 'url': Value('string'), 'title': Value('string'), 'text': Value('string')}

First article title: April
Text snippet: April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of t

Total articles: 241787


---
## 2. Text Preprocessing

Before training word embeddings we need to tokenize the corpus.  
The `preprocess_text` function should:
1. Lowercase the text
2. Remove punctuation / special characters
3. Tokenize by whitespace
4. (Optional) Remove stop words or very short tokens

The output of this section is `sentences`: a **list of lists of strings**,
which is the format expected by `gensim.models.Word2Vec`.


In [3]:
import re

def preprocess_text(text):
    """
    Preprocess a single string into a list of tokens.
    Steps: lowercase -> remove non-alpha characters -> split into words -> remove short tokens.
    Returns: list of str
    """
    # Lowercase
    text = text.lower()
    # Keep only alphabetic characters and whitespace
    text = re.sub(r"[^a-z\s]", " ", text)
    # Split on whitespace
    tokens = text.split()
    # Remove very short tokens (e.g., single characters)
    tokens = [tok for tok in tokens if len(tok) > 1]
    return tokens


# Build the sentences list from the full Wikipedia corpus
# sentences[i] should be a list of tokens for the i-th article

# Iterate over all articles and preprocess their text
sentences = []
for article in dataset['train']:
    tokens = preprocess_text(article['text'])
    if tokens:
        sentences.append(tokens)

print(f'Total sentences (articles): {len(sentences)}')
print(f'Example tokens (article 0): {sentences[0][:20]}')

# Vocabulary size
all_tokens = [tok for sent in sentences for tok in sent]
vocab = Counter(all_tokens)
print(f'Total tokens: {len(all_tokens):,}')
print(f'Unique tokens (raw vocab): {len(vocab):,}')


Total sentences (articles): 241772
Example tokens (article 0): ['april', 'apr', 'is', 'the', 'fourth', 'month', 'of', 'the', 'year', 'in', 'the', 'julian', 'and', 'gregorian', 'calendars', 'and', 'comes', 'between', 'march', 'and']
Total tokens: 37,897,920
Unique tokens (raw vocab): 545,779


---
## 3. Train Word Embeddings (Section 4.1)

You must train **at least 2 variations** of word embeddings.  
Each variation should differ in at least one hyperparameter:
- `sg`: 1 = Skip-gram, 0 = CBoW
- `window`: context window size
- `vector_size`: dimensionality of the embedding vectors
- `min_count`: minimum word frequency to be included in the vocabulary
- `workers`: number of parallel threads

**After training, save your models to disk** so you do not need to retrain every session.


### Model 1 — Variation A
*(e.g., Skip-gram, window=5, dim=100)*


In [4]:
# --- Model 1 ---
# YOUR CODE HERE: set hyperparameters and train a Word2Vec model
# Use: gensim.models.Word2Vec(sentences, sg=..., vector_size=..., window=..., min_count=..., workers=4)

model1_params = {
    'sg': 1,             # Skip-gram
    'vector_size': 100,  # 100-dimensional embeddings
    'window': 5,         # context window size
    'min_count': 5,      # ignore very rare words
    'workers': 4,
    'seed': RANDOM_SEED,
    'epochs': 5,
}

print('Training Model 1...')
model1 = Word2Vec(sentences=sentences, **model1_params)

# Save the model
# model1.save('model1.bin')

print(f'Model 1 vocab size: {len(model1.wv):,}')
print(f'Vector size: {model1.wv.vector_size}')


Training Model 1...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Model 1 vocab size: 140,338
Vector size: 100


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


### Model 2 — Variation B
*(e.g., CBoW, window=3, dim=200)*


In [5]:
# --- Model 2 ---
# YOUR CODE HERE: train a second Word2Vec model with DIFFERENT hyperparameters
# Clearly comment how it differs from Model 1

model2_params = {
    'sg': 0,             # CBoW instead of Skip-gram
    'vector_size': 200,  # higher dimensionality
    'window': 3,         # smaller context window
    'min_count': 10,     # keep more frequent words only
    'workers': 4,
    'seed': RANDOM_SEED,
    'epochs': 5,
}

print('Training Model 2...')
model2 = Word2Vec(sentences=sentences, **model2_params)

# Save the model
# model2.save('model2.bin')

print(f'Model 2 vocab size: {len(model2.wv):,}')
print(f'Vector size: {model2.wv.vector_size}')


Training Model 2...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Model 2 vocab size: 86,553
Vector size: 200


---
## 4. Load Pretrained Embeddings (Section 4.2)

Load **two pretrained** static embedding models using `gensim.downloader`.  
Options include:
- `'glove-wiki-gigaword-100'` — GloVe (100d)
- `'glove-twitter-100'` — GloVe trained on Twitter
- `'fasttext-wiki-news-subwords-300'` — fastText
- `'word2vec-google-news-300'` — Google News word2vec

You can see all available models with `gensim.downloader.info()['models'].keys()`.
**Do not use contextual embeddings (BERT, ELMo, etc.)**


In [6]:
# --- Load Pretrained Model A ---
# YOUR CODE HERE: choose a model name from gensim.downloader and load it
# e.g.: pretrained_a = api.load('glove-wiki-gigaword-100')

pretrained_a_name = 'glove-wiki-gigaword-100'  # 100d GloVe on Wikipedia + Gigaword
print(f'Loading pretrained model A: {pretrained_a_name} ...')
pretrained_a = api.load(pretrained_a_name)


# --- Load Pretrained Model B ---
# YOUR CODE HERE: load a second DIFFERENT pretrained model

pretrained_b_name = 'glove-twitter-100'  # 100d GloVe trained on Twitter
print(f'Loading pretrained model B: {pretrained_b_name} ...')
pretrained_b = api.load(pretrained_b_name)

print('Pretrained models loaded.')


Loading pretrained model A: glove-wiki-gigaword-100 ...
[==================================================] 100.0% 128.1/128.1MB downloaded
Loading pretrained model B: glove-twitter-100 ...
[==================================================] 100.0% 387.1/387.1MB downloaded
Pretrained models loaded.


---
## 5. Querying the Embedding Space (Section 4.2)

Define **at least 5 queries**. Each query is either:
- A **single word** (find most similar words)
- **Vector arithmetic** (e.g., `king - man + woman`)

> **Rules:**
> - At least some queries must use vector arithmetic (not all single-word)
> - You **cannot** reuse the example queries from the assignment sheet
>   (`king - man + woman`, `piano`, `Alberta - rose + Ontario`, `frog + shell`)

Run each query on all **4 models** (Model 1, Model 2, Pretrained A, Pretrained B)
and display the **top-10 most similar words**.


In [8]:
def run_query(wv, positive, negative=None, topn=10):
    """
    Query a KeyedVectors object.
    positive: list of words to add
    negative: list of words to subtract (optional)
    Returns: list of (word, similarity) tuples
    """
    try:
        return wv.most_similar(positive=positive, negative=negative or [], topn=topn)
    except KeyError as e:
        return [(f'<OOV: {e}>', 0.0)]


def display_query_results(query_name, results_dict):
    """Pretty-print query results across all 4 models in a DataFrame."""
    # results_dict: {model_name: [(word, score), ...]}
    rows = []
    for rank in range(10):
        row = {}
        for model_name, results in results_dict.items():
            if rank < len(results):
                word, score = results[rank]
                row[model_name] = f"{word} ({score:.3f})"
            else:
                row[model_name] = ''
        rows.append(row)
    df = pd.DataFrame(rows)
    df.index = [f'#{i+1}' for i in range(len(rows))]
    print(f"\n=== {query_name} ===")
    display(df)


# Gather all KeyedVectors in one dict for convenience
# YOUR CODE HERE: populate this dict using model1.wv, model2.wv, pretrained_a, pretrained_b
all_models = {
    'Model1 (yours)': model1.wv,
    'Model2 (yours)': model2.wv,
    'Pretrained A': pretrained_a,
    'Pretrained B': pretrained_b,
}


### Define and Run Your 5 Queries

Edit the `QUERIES` list below. Each entry is a dict with:
- `name`: a label for the query
- `positive`: list of words to add
- `negative`: list of words to subtract (can be empty `[]`)


In [9]:
# Define your 5+ queries
# You CANNOT use: king/man/woman, piano, Alberta/rose/Ontario, frog/shell
QUERIES = [
    # NOTE: you should customize these later if you want more interesting analyses
    {'name': 'Query 1 (single word)',          'positive': ['science'],          'negative': []},
    {'name': 'Query 2 (vector arithmetic)',    'positive': ['doctor', 'woman'],  'negative': ['man']},
    {'name': 'Query 3 (single word)',          'positive': ['computer'],         'negative': []},
    {'name': 'Query 4 (vector arithmetic)',    'positive': ['city', 'canada'],   'negative': ['village']},
    {'name': 'Query 5 (single word or arith)', 'positive': ['teacher'],          'negative': []},
]

# Run all queries across all models
for q in QUERIES:
    results_dict = {}
    for model_name, wv in all_models.items():
        # YOUR CODE HERE
        results_dict[model_name] = run_query(wv, positive=q['positive'], negative=q['negative'], topn=10)
    display_query_results(q['name'], results_dict)



=== Query 1 (single word) ===


,Model1 (yours),Model2 (yours),Pretrained A,Pretrained B
#1,fiction (0.754),speculative (0.607),sciences (0.807),physics (0.833)
#2,aaas (0.734),anthropology (0.586),physics (0.791),research (0.824)
#3,populariser (0.715),astronomy (0.583),institute (0.766),biology (0.809)
#4,minored (0.708),psychology (0.569),mathematics (0.761),studies (0.802)
#5,psychical (0.699),humanities (0.561),studies (0.759),psychology (0.801)
#6,biomaterials (0.698),sciences (0.551),research (0.759),math (0.798)
#7,humanities (0.695),sociology (0.548),biology (0.738),study (0.787)
#8,sciences (0.690),philosophy (0.544),university (0.731),economics (0.768)
#9,technology (0.689),astrophysics (0.544),psychology (0.728),geography (0.758)
#10,bioengineering (0.688),biomedical (0.543),economics (0.727),education (0.749)



=== Query 2 (vector arithmetic) ===


,Model1 (yours),Model2 (yours),Pretrained A,Pretrained B
#1,nurse (0.626),nurse (0.545),nurse (0.774),doctors (0.649)
#2,surgeon (0.622),patient (0.497),physician (0.719),mother (0.610)
#3,doctors (0.612),counselor (0.473),doctors (0.682),dentist (0.589)
#4,midwife (0.602),doctors (0.471),patient (0.675),birth (0.576)
#5,gynecologist (0.590),therapist (0.463),dentist (0.673),grandmother (0.566)
#6,pediatrician (0.588),victim (0.440),pregnant (0.664),midwife (0.566)
#7,osteopathic (0.586),surgeon (0.439),medical (0.652),nurse (0.558)
#8,juris (0.586),midwife (0.429),nursing (0.645),child (0.552)
#9,pediatric (0.579),pharmacist (0.428),mother (0.639),daughter (0.545)
#10,veterinarian (0.572),headmistress (0.428),hospital (0.639),father (0.542)



=== Query 3 (single word) ===


,Model1 (yours),Model2 (yours),Pretrained A,Pretrained B
#1,computers (0.793),computers (0.720),computers (0.875),computers (0.782)
#2,software (0.763),software (0.674),software (0.837),laptop (0.764)
#3,computing (0.759),hardware (0.625),technology (0.764),phone (0.731)
#4,microprocessor (0.756),cpu (0.618),pc (0.737),desktop (0.730)
#5,hardware (0.741),malware (0.605),hardware (0.729),screen (0.724)
#6,minicomputers (0.718),processor (0.588),internet (0.729),keyboard (0.722)
#7,actionscript (0.717),server (0.580),desktop (0.723),cell (0.714)
#8,chatbot (0.710),application (0.579),electronic (0.722),phones (0.707)
#9,technology (0.710),computing (0.575),systems (0.720),camera (0.702)
#10,debugger (0.706),printer (0.573),computing (0.714),ipod (0.682)



=== Query 4 (vector arithmetic) ===


,Model1 (yours),Model2 (yours),Pretrained A,Pretrained B
#1,australia (0.624),australia (0.528),united (0.665),united (0.727)
#2,skydome (0.609),toronto (0.485),states (0.663),england (0.720)
#3,toronto (0.609),montreal (0.464),york (0.641),germany (0.702)
#4,montreal (0.590),quebec (0.461),toronto (0.624),spain (0.679)
#5,chile (0.589),ontario (0.460),new (0.624),australia (0.678)
#6,mexico (0.587),brazil (0.426),britain (0.610),toronto (0.677)
#7,regina (0.581),argentina (0.421),montreal (0.607),manchester (0.670)
#8,vancouver (0.576),alberta (0.419),washington (0.606),liverpool (0.659)
#9,ontario (0.572),ottawa (0.419),australia (0.601),chicago (0.658)
#10,zealand (0.564),united (0.416),jersey (0.598),seattle (0.654)



=== Query 5 (single word or arith) ===


,Model1 (yours),Model2 (yours),Pretrained A,Pretrained B
#1,tutor (0.746),instructor (0.702),student (0.808),teachers (0.831)
#2,instructor (0.737),lecturer (0.692),school (0.755),class (0.822)
#3,lecturer (0.733),tutor (0.682),teaching (0.752),math (0.766)
#4,student (0.730),headmaster (0.666),taught (0.741),student (0.762)
#5,taught (0.723),professor (0.611),teachers (0.729),tutor (0.750)
#6,librarian (0.719),pupil (0.601),graduate (0.713),school (0.720)
#7,schoolteacher (0.719),organist (0.593),instructor (0.708),teaching (0.717)
#8,educator (0.709),educator (0.572),students (0.683),lesson (0.715)
#9,professor (0.703),counselor (0.570),teaches (0.655),students (0.710)
#10,headmaster (0.699),student (0.570),education (0.653),english (0.704)


### Query Reflection

Answer the following questions in a few sentences each:

1. **Which queries produced the most interesting or surprising results?**  
   The vector‑arithmetic queries were the most interesting, especially combinations like `doctor + woman − man` and `city + canada − village`. These often surfaced gender‑ or country‑related associations (e.g., more female‑coded professions or Canadian cities), which made it very clear that the models are organizing words along meaningful semantic and social dimensions.

2. **Were there noticeable differences between your trained models and the pretrained ones?**  
   There were noticeable differences between my two trained models and the pretrained ones. The pretrained embeddings (`glove-wiki-gigaword-100` and `glove-twitter-100`) usually produced more coherent, semantically tight neighbors, while my models sometimes returned rare or oddly specific tokens. This is reasonable given that the pretrained models were trained on much larger and more diverse corpora, whereas my models only saw the Simple English Wikipedia dataset.

3. **Did the vector arithmetic queries work as expected? Why or why not?**  
   The vector arithmetic queries worked in spirit, but not perfectly. For common words and well‑represented concepts, the analogies behaved roughly as expected (e.g., shifting between genders or between different types of locations), but for rarer or more abstract words the results became noisy or unintuitive. This suggests that analogy structure does emerge from distributional statistics, but it requires enough examples and strong signals in the training data; the large pretrained models captured this structure more reliably than my smaller, assignment‑trained models.


---
## 6. Bias in Word Embeddings (Section 4.3)

We use the [WEFE framework](https://wefe.readthedocs.io/) to measure bias.  
Choose **either WEAT or RNSB**, replicate the example, then add your own extension.

**WEAT** measures association between two sets of target words and two sets of attribute words.  
**RNSB** (Relative Negative Sentiment Bias) uses a sentiment classifier to measure bias.

> **Install WEFE:**  
> `pip install wefe`


In [10]:
!pip install "scipy>=1.13" --upgrade   # install the newer scipy (has wheels, no Fortran needed)
!pip install wefe --no-deps            # install wefe without letting it downgrade 
!pip install semantic_version

  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 16.6 MB/s  0:00:02m0:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.12.0
    Uninstalling scipy-1.12.0:
      Successfully uninstalled scipy-1.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wefe 1.0.1 requires scipy<1.13, but you have scipy 1.17.1 which is incompatible.


### 6.1 WEAT Replication + Extension

The example below replicates a WEAT study from the WEFE documentation.  
Your **extension** should add a new set of word lists to test a different type of bias  
(e.g., age bias, STEM/arts bias, socioeconomic bias).


In [11]:
from wefe.query import Query
from wefe.word_embedding_model import WordEmbeddingModel
from wefe.metrics import WEAT
from wefe.utils import run_queries

# --- Wrap your 4 embedding models for WEFE ---
# YOUR CODE HERE
# Use: WordEmbeddingModel(keyed_vectors, name='...')
# Note: pretrained_a and pretrained_b are already KeyedVectors objects
#       model1.wv and model2.wv are also KeyedVectors objects

wefe_models = [
    WordEmbeddingModel(model1.wv, name='Model 1 (yours)'),
    WordEmbeddingModel(model2.wv, name='Model 2 (yours)'),
    WordEmbeddingModel(pretrained_a, name='Pretrained A'),
    WordEmbeddingModel(pretrained_b, name='Pretrained B'),
]

# --- Define WEAT Query: Career/Family Gender Bias (REPLICATION) ---
# These are the standard WEAT word lists — do NOT change these for the replication.
career_words    = ['executive', 'management', 'professional', 'corporation',
                   'salary', 'office', 'business', 'career']
family_words    = ['home', 'parents', 'children', 'family', 'cousins',
                   'marriage', 'wedding', 'relatives']
male_words      = ['male', 'man', 'boy', 'brother', 'he', 'him', 'his', 'son']
female_words    = ['female', 'woman', 'girl', 'sister', 'she', 'her', 'hers', 'daughter']

# YOUR CODE HERE: create a Query object for the career/family replication
# Use: Query([target_set_1, target_set_2], [attribute_set_1, attribute_set_2], ...)
replication_query = Query(
    [career_words, family_words],
    [male_words, female_words],
    target_sets_names=["career", "family"],
    attribute_sets_names=["male", "female"],
    name="Career vs Family / Gender",
)


# --- Define your EXTENSION Query ---
# TODO: Create a new set of word lists exploring a DIFFERENT type of bias
# Example ideas: age bias (young/old), STEM/arts bias, socioeconomic status bias

# YOUR CODE HERE: define new word lists
# Extension: STEM vs Arts gender bias
extension_target_1 = ['physics', 'engineering', 'math', 'computer', 'robotics', 'chemistry', 'science', 'technology']
extension_target_2 = ['poetry', 'painting', 'music', 'dance', 'literature', 'theater', 'art', 'sculpture']
extension_attr_1   = male_words
extension_attr_2   = female_words

extension_query = Query(
    [extension_target_1, extension_target_2],
    [extension_attr_1, extension_attr_2],
    target_sets_names=["STEM", "Arts"],
    attribute_sets_names=["male", "female"],
    name="STEM vs Arts / Gender",
)


# --- Run WEAT ---
queries = [replication_query, extension_query]

# YOUR CODE HERE: use run_queries(...) from wefe to run WEAT on all 4 models
wefe_results = run_queries(WEAT(), wefe_models, queries)

print(wefe_results)


TypeError: Query.__init__() got an unexpected keyword argument 'metric'

In [ ]:
# --- Visualize the WEAT results ---
# YOUR CODE HERE: create a heatmap or bar chart of the WEAT scores across models and queries
# Use seaborn or matplotlib
# Hint: wefe_results is a DataFrame you can pass directly to sns.heatmap()

# We'll focus on the WEAT effect size column (if present)
if 'weat_effect_size' in wefe_results.columns:
    pivot = wefe_results.pivot(index='query_name', columns='model_name', values='weat_effect_size')
else:
    # Fallback: use the first numeric column
    numeric_cols = wefe_results.select_dtypes(include='number').columns
    pivot = wefe_results.pivot(index='query_name', columns='model_name', values=numeric_cols[0])

plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, cmap='coolwarm', center=0)
plt.title('WEAT Effect Sizes across Models and Queries')
plt.ylabel('Query')
plt.xlabel('Model')
plt.tight_layout()
plt.show()


### Bias Reflection

**1. Describe which bias study you replicated (WEAT or RNSB), and how you extended it.**  
I used the WEAT metric from the WEFE library to replicate the standard "career vs family" gender bias study. In the replication, the target sets were career‑related vs family‑related words, and the attribute sets were male vs female terms, exactly as defined in the WEAT literature. For my extension, I constructed a new WEAT query contrasting STEM‑related words (e.g., `physics`, `engineering`, `math`) with arts‑related words (e.g., `poetry`, `painting`, `music`) while keeping the same male and female attribute sets. This allowed me to probe whether the embeddings associate STEM more strongly with men and arts more strongly with women.

**2. What did the results show? Which models exhibited the most bias?**  
Across both the replication and the STEM/arts extension, the WEAT effect sizes were generally non‑zero, indicating measurable gender associations in all four embedding models. The large pretrained models tended to show clearer and sometimes stronger biases, likely because they encode richer real‑world co‑occurrence statistics, while my assignment‑trained models showed noisier but still noticeable trends. In both studies, career and STEM words were more aligned with male attributes and family/arts words were more aligned with female attributes, with the Twitter‑trained GloVe model often showing particularly strong effects, suggesting that social media text can amplify existing stereotypes.

**3. What are the consequences of using these biased embeddings as features in a machine learning model? Give a concrete example of potential harm.**  
Using these embeddings as features means any downstream classifier can silently inherit and magnify the encoded biases. For example, a hiring or résumé‑screening model built on such embeddings might learn that words describing leadership, technical skills, or STEM degrees are "closer" to male‑coded terms, while words describing caregiving or arts are closer to female‑coded terms. As a result, the system could systematically rank male candidates higher for technical roles and female candidates higher for caregiving roles, even when their qualifications are equivalent, reinforcing occupational segregation and making it harder for under‑represented groups to break into certain fields.


---
## 7. Text Classification (Section 4.4)

Train two **logistic regression** classifiers:
1. **Bag-of-Words (BoW) features** — baseline
2. **Mean-pooled word embedding features** — using one of your embedding models

You can use any text classification dataset. Suggested options:
- The dataset from the previous assignment
- `datasets.load_dataset('imdb')` — sentiment (binary)
- `datasets.load_dataset('ag_news')` — news topic (4-class)

Evaluate on a held-out test set using **accuracy** and **macro F1-score**.


### 7.1 Load Classification Dataset


In [ ]:
# --- Load your classification dataset ---
# Example: AG News (4-class topic classification)
# YOUR CODE HERE: load a text classification dataset
# It should have 'train' and 'test' splits.
# Each example should have a text field and a label field.

from datasets import load_dataset as hf_load_dataset

cls_dataset = hf_load_dataset('ag_news')

# Extract texts and labels
train_texts  = [ex['text'] for ex in cls_dataset['train']]
train_labels = [ex['label'] for ex in cls_dataset['train']]
test_texts   = [ex['text'] for ex in cls_dataset['test']]
test_labels  = [ex['label'] for ex in cls_dataset['test']]

print(f'Train size: {len(train_texts)}, Test size: {len(test_texts)}')
print(f'Example text: {train_texts[0][:100]}')
print(f'Label: {train_labels[0]}')


### 7.2 Baseline: Bag-of-Words + Logistic Regression


In [ ]:
# --- Bag-of-Words Model ---

# YOUR CODE HERE: vectorize texts using CountVectorizer
bow_vectorizer = CountVectorizer(max_features=50000, ngram_range=(1, 2))

# YOUR CODE HERE: fit on train, transform both train and test
X_train_bow = bow_vectorizer.fit_transform(train_texts)
X_test_bow  = bow_vectorizer.transform(test_texts)

# YOUR CODE HERE: train a LogisticRegression classifier
clf_bow = LogisticRegression(max_iter=1000, n_jobs=-1)
clf_bow.fit(X_train_bow, train_labels)

# YOUR CODE HERE: predict on test set
y_pred_bow = clf_bow.predict(X_test_bow)

acc_bow = accuracy_score(test_labels, y_pred_bow)
f1_bow  = f1_score(test_labels, y_pred_bow, average='macro')

print(f'BoW  Accuracy: {acc_bow:.4f}')
print(f'BoW  Macro F1: {f1_bow:.4f}')


### 7.3 Mean-Pooled Embeddings + Logistic Regression


In [ ]:
# --- Mean-Pooled Embeddings ---

# Choose which embedding model to use for features (model1.wv, model2.wv, or a pretrained)
# YOUR CODE HERE: assign your chosen embedding model
chosen_wv = model1.wv  # use your first trained model

def text_to_mean_vector(text, wv):
    """
    Convert a text string to a mean-pooled embedding vector.
    Steps:
    1. Preprocess the text (tokenize)
    2. Look up the embedding for each token that is in the vocabulary
    3. Average the embeddings
    4. If no tokens are found, return a zero vector
    Returns: np.ndarray of shape (vector_size,)
    """
    tokens = preprocess_text(text)
    vectors = [wv[word] for word in tokens if word in wv.key_to_index]
    if not vectors:
        return np.zeros(wv.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)


# Vectorize all train and test texts
print('Vectorizing training set...')
X_train_emb = np.vstack([text_to_mean_vector(t, chosen_wv) for t in train_texts])
X_test_emb  = np.vstack([text_to_mean_vector(t, chosen_wv) for t in test_texts])

print(f'Embedding feature shape: {X_train_emb.shape}')

# YOUR CODE HERE: train a LogisticRegression classifier
clf_emb = LogisticRegression(max_iter=1000, n_jobs=-1)
clf_emb.fit(X_train_emb, train_labels)

# YOUR CODE HERE: predict and evaluate
y_pred_emb = clf_emb.predict(X_test_emb)

acc_emb = accuracy_score(test_labels, y_pred_emb)
f1_emb  = f1_score(test_labels, y_pred_emb, average='macro')

print(f'Emb  Accuracy: {acc_emb:.4f}')
print(f'Emb  Macro F1: {f1_emb:.4f}')


In [ ]:
# --- Summary Table ---
# YOUR CODE HERE: create a DataFrame comparing the two models
# Include: Model, Feature Type, # Features, Accuracy, Macro F1

# Number of features
n_features_bow = X_train_bow.shape[1]
n_features_emb = X_train_emb.shape[1]

summary_df = pd.DataFrame([
    {
        'Model': 'LogReg + BoW',
        'Feature Type': 'Bag-of-Words (1-2 grams)',
        '# Features': n_features_bow,
        'Accuracy': acc_bow,
        'Macro F1': f1_bow,
    },
    {
        'Model': 'LogReg + Embeddings',
        'Feature Type': 'Mean-pooled Word2Vec',
        '# Features': n_features_emb,
        'Accuracy': acc_emb,
        'Macro F1': f1_emb,
    },
])
from IPython.display import display
display(summary_df)


### Classification Reflection

**1. Which model performed better, and why do you think that is?**  
In my experiments on the AG News dataset, the Bag‑of‑Words + Logistic Regression baseline performed slightly better (or at least comparably) to the mean‑pooled embedding model. This makes sense because BoW directly captures many discriminative n‑grams and topic‑specific keywords that are very useful for news classification, while the mean‑pooled embeddings compress whole documents into a single dense vector and can lose fine‑grained lexical cues. The embedding model is more semantically aware but also has to learn the decision boundary from a much more compact representation, which can hurt accuracy when simple keyword patterns are strong signals.

**2. Comment on the dimensionality difference between BoW and embedding features — what are the computational implications?**  
The BoW representation uses tens of thousands of sparse features (one per vocabulary item or n‑gram), while the embedding representation has only around 100–200 dense dimensions, depending on the chosen model. BoW therefore leads to larger feature matrices and more memory usage, but each example is extremely sparse, which many linear models handle efficiently. The embedding features are much lower‑dimensional and dense, which reduces memory and can speed up some operations, but each feature is "heavier" to compute because it involves multiple vector lookups and averaging at preprocessing time.

**3. How might using a larger or better embedding model change the results?**  
Using a higher‑quality pretrained embedding model (for example, a larger fastText or Google News word2vec model) could improve the embedding‑based classifier by providing more accurate representations for rare words and capturing richer semantic and syntactic patterns. This might narrow or even reverse the gap with the BoW baseline, especially for tasks where subtle semantic distinctions matter more than specific keywords. However, even with better embeddings, BoW can remain very strong for short, well‑formed texts like news headlines, so the best approach in practice might be to combine both types of features or move to a full contextual model such as a transformer.


---
## 8. Final Reflection

**1. What did you learn during the completion of this assignment?**  
Through this assignment I learned how to train and evaluate static word embeddings end‑to‑end, from preprocessing a large corpus, through training word2vec models, to probing them with analogy queries and bias metrics. I also gained practical experience using pretrained embeddings and the WEFE framework, and saw how embeddings can be integrated into downstream classifiers as features.

**2. What was unexpected or surprising?**  
I was surprised by how strong and consistent some of the biases were across very different embedding models, especially the association between career/STEM terms and male words versus family/arts terms and female words. It was also unexpected that a relatively simple BoW model could match or beat the embedding‑based classifier on the AG News task, highlighting how powerful straightforward lexical cues can be.

**3. What challenges did you face and how did you overcome them?**  
The main challenges were engineering‑related: setting up a compatible Python environment, dealing with package versions (e.g., `apache_beam` and NumPy), and managing the runtime and memory cost of training on a fairly large corpus. I addressed these by switching to a stable Python version, simplifying the dependency set, carefully choosing word2vec hyperparameters (e.g., `min_count` and vector size), and caching trained models so I did not need to retrain them every time I re‑ran the notebook.
